<div style="
    background-color: #7BAFD4;
    border-radius: 18px;
    padding: 22px 26px;
    display: flex;
    align-items: center;
    gap: 24px;
    margin-bottom: 18px;
">
    <img src="592_avatar_lr.png" alt="GEOG 592"
        style="width:110px;height:110px;border-radius:22px;">
    <div>
        <div style="
            font-family: Impact, 'Arial Black', Arial, sans-serif;
            font-size: 42px;
            font-weight: 900;
            letter-spacing: 2px;
            line-height: 1;
            color: white;
        ">GEOG 592</div>
        <div style="
            font-family: Arial, Helvetica, sans-serif;
            font-size: 22px;
            font-weight: 700;
            color: #13294B;
            margin-top: 8px;
        ">GIS Programming</div>
        <div style="
            font-family: Arial, Helvetica, sans-serif;
            font-size: 16px;
            color: #13294B;
            margin-top: 6px;
        ">Cartography examples using GBIF and census data</div>
    </div>
</div>



In [ ]:
### Code mostly written by AI 
## This cell might take some time to execute
# https://www.gbif.org/

import time
import requests
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt



species = ["Morus alba", "Morus rubra"]
year = 2025

api = "https://api.gbif.org/v1"
records = []

for name in species:

    # 1. Find the GBIF taxon identifier for the scientific name.
    response = requests.get(
        f"{api}/species/match",
        params={"name": name, "strict": "true"},
        timeout=60
    )
    response.raise_for_status()
    match = response.json()

    if (
        match.get("matchType") != "EXACT"
        or match.get("rank") != "SPECIES"
    ):
        raise ValueError(f"Check the species name: {name}")

    taxon_key = match.get("acceptedUsageKey", match["usageKey"])

    # 2. Define which occurrence records to request.
    parameters = {
        "taxonKey": taxon_key,
        "country": "US",
        # "stateProvince": "North Carolina",
        "year": year,
        "basisOfRecord": "HUMAN_OBSERVATION",
        "occurrenceStatus": "PRESENT",
        "hasCoordinate": "true",
        "hasGeospatialIssue": "false",
        "limit": 300,
        "offset": 0
    }

    # 3. Retrieve successive pages of records.
    species_records = []

    while True:
        response = requests.get(
            f"{api}/occurrence/search",
            params=parameters,
            timeout=60
        )
        response.raise_for_status()
        page = response.json()

        # LIMIT THE NUMBER OF CASES (you can chage it but, the larger the number...the longer the wait)
        if page["count"] > 9000:
            raise ValueError(
                f"{name} has too many records for this example. "
                "Narrow the date range or use a GBIF bulk download."
            )

        for record in page["results"]:
            record["queried_species"] = name

        species_records.extend(page["results"])

        if page.get("endOfRecords") or not page["results"]:
            break

        parameters["offset"] += len(page["results"])
        time.sleep(0.2)

    records.extend(species_records)
    print(f"{name}: {len(species_records):,} records")


In [ ]:
mulberries = pd.DataFrame(records)
mulberries = mulberries[["queried_species", "decimalLongitude", "decimalLatitude"]] ## select only the columns that I care about. 
mulberries.head()

In [ ]:
# getting the US census data

url_0 = "https://www2.census.gov/geo/tiger/GENZ2024/shp/" # url for census tiger data 
url_county = "cb_2024_us_county_500k.zip" # county level
url_state = "cb_2024_us_state_500k.zip" # state level 


### Code for county level data 

# counties = gpd.read_file(url_0+url_county)
# nc_counties = counties.loc[counties["STATEFP"] == "37"].copy()
# nc_counties.plot(edgecolor="black", facecolor="white")

## remove Alaska, Hawaii and those other islands which makes mapping difficult, but life much nicer.  
states = gpd.read_file(url_0+url_state)
contiguous_us = states.loc[
    ~states["STUSPS"].isin(["AK", "HI", "PR", "VI", "GU", "MP", "AS"])
]

contiguous_us = contiguous_us.to_crs(epsg=5070)
contiguous_us.plot(edgecolor="black", facecolor="white")



In [ ]:

## make a gdp from the x and y data. 
mulberry_points = gpd.GeoDataFrame(
    mulberries.copy(),
    geometry=gpd.points_from_xy( ## the method is calleed points_from_xy
        mulberries["decimalLongitude"], # x
        mulberries["decimalLatitude"] # y
        ),
    crs="EPSG:4326" # often the crs is in 4326 if it is a lat lon coordinate
    )

mulberry_points = mulberry_points.to_crs(epsg=5070)


fig, ax = plt.subplots(figsize=(12, 6))
contiguous_us.plot(
    ax=ax,
    facecolor="whitesmoke",
    edgecolor="gray",
    linewidth=0.5
)

colors = {"Morus alba": "purple", "Morus rubra": "red"}

for species, color in colors.items():
    mulberry_points.loc[
        mulberry_points["queried_species"] == species
    ].plot(
        ax=ax,
        color=color,
        markersize=15,
        alpha=0.7,
        label=species
    )

ax.set_title("Mulberry observations")
ax.legend(title="Species")
ax.set_axis_off()

plt.show()

In [ ]:
# select points within contiguous states and keep the NAME of the state. 
points_by_state = gpd.sjoin(
    mulberry_points,
    contiguous_us[["NAME", "geometry"]],
    how="inner",
    predicate="within"
).drop(columns="index_right")

points_by_state.head()

In [ ]:
### Plotting code suggested by Ai
fig, ax = plt.subplots(figsize=(12, 6)) # creating the fig and ax objects using plt
contiguous_us.plot(
    ax=ax,
    facecolor="whitesmoke",
    edgecolor="gray",
    linewidth=0.5
)

colors = {"Morus alba": "purple", "Morus rubra": "red"} # making a dictionary 

for species, color in colors.items(): # 
    points_by_state.loc[
        points_by_state["queried_species"] == species
    ].plot(
        ax=ax,
        color=color,
        markersize=15,
        alpha=0.7,
        label=species
    )

ax.set_title("Mulberry observations")
ax.legend(title="Species")
ax.set_axis_off()

plt.show()

In [ ]:
def pointMap(ax):
    ### Plotting code suggested by AI

    # fig, ax = plt.subplots(figsize=(12, 6)) # creating the fig and ax objects using plt
    contiguous_us.plot(
        ax=ax,
        facecolor="whitesmoke",
        edgecolor="gray",
        linewidth=0.5
    )

    colors = {"Morus alba": "purple", "Morus rubra": "red"} # making a dictionary 

    for species, color in colors.items(): # 
        points_by_state.loc[
            points_by_state["queried_species"] == species
        ].plot(
            ax=ax,
            color=color,
            markersize=15,
            alpha=0.7,
            label=species
        )

    ax.set_title("Mulberry observations")
    ax.legend(title="Species")
    ax.set_axis_off()
    return ax
  


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6)) 
ax =  pointMap(ax)
plt.show()

In [ ]:
counts = (
    points_by_state
    .groupby(["NAME", "queried_species"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["Morus rubra", "Morus alba"], fill_value=0)
)

# counts.head()

In [ ]:
def barPlot(ax, counts):
    counts = counts.loc[
        counts.sum(axis=1).sort_values(ascending=False).index
    ]

    counts[["Morus rubra", "Morus alba"]].plot.bar(
        ax=ax,
        stacked=True,
        color=["firebrick", "mediumpurple"]
    )

    ax.set_xlabel("State")
    ax.set_ylabel("Number of observation records")
    ax.set_title("Red and white mulberry observations by state")
    ax.legend(["Red mulberry", "White mulberry"])
    ax.tick_params(axis="x", labelrotation=90)

    return ax



In [ ]:
fig, ax = plt.subplots(figsize=(12, 4)) 
ax = barPlot(ax,counts)
fig.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(
    2, 1,
    figsize=(14, 12),
    layout="constrained"
)

pointMap(axes[0])
barPlot(axes[1], counts)

plt.show()

In [ ]:
fig = plt.figure(figsize=(14, 8))

# [left, bottom, width, height], as fractions of the figure.
position_bar = [0.1, 0.15, 0.8, 0.75]
position_map= [0.20, 0.15, 0.8, 0.75]

ax_bar = fig.add_axes(position_bar, zorder=1)
barPlot(ax_bar, counts)

ax_map = fig.add_axes(position_map, zorder=2)
pointMap(ax_map)  # Match your function's arguments.

legend = ax_map.get_legend() # removing the legend
if legend is not None:
    legend.remove()

ax_map.set_title("") # removing the title 

# Make the upper layer transparent.
ax_map.set_facecolor("none")
ax_map.patch.set_alpha(0)
ax_map.set_axis_off()

plt.show()

In [ ]:
# Export the figure as png and as pdf
fig.savefig("mulberry_figure.png", dpi=300, bbox_inches="tight")
fig.savefig("mulberry_figure.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Made with AI

def choroplethMap(ax, contiguous_us, counts):
    # Total observations per state.
    totals = counts.sum(axis=1).rename("total")

    # Join totals to the state polygons.
    state_map = contiguous_us.merge(
        totals,
        left_on="NAME",
        right_index=True,
        how="left"
    )

    # States without records receive zero.
    state_map["total"] = state_map["total"].fillna(0)

    state_map.plot(
        column="total",
        cmap="RdPu", # This color was selected by me not Ai 
        edgecolor="gray",
        linewidth=0.5,
        legend=True,
        legend_kwds={"label": "Number of observation records"},
        ax=ax
    )

    ax.set_title("Mulberry observations by state")
    ax.set_axis_off()

    return ax

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

choroplethMap(ax, contiguous_us, counts)

plt.show()

# How many mulberries are in Delaware?  

In [ ]:
#%pip install mapclassify
## let's do some choropleths with classified data
import mapclassify


In [ ]:
## code that I generated with AI to make the three different classification schemes

def choroplethEqual(ax, contiguous_us, counts):
    totals = counts.sum(axis=1).rename("total")

    state_map = contiguous_us.merge(
        totals, left_on="NAME", right_index=True, how="left"
    )
    state_map["total"] = state_map["total"].fillna(0)

    state_map.plot(
        column="total",
        scheme="equalinterval",
        k=5,
        cmap="RdPu",
        edgecolor="gray",
        linewidth=0.5,
        legend=True,
        legend_kwds={"title": "Observation records", "fontsize": 8},
        ax=ax
    )

    ax.set_title("Equal interval")
    ax.set_axis_off()
    return ax

def choroplethQuantiles(ax, contiguous_us, counts):
    totals = counts.sum(axis=1).rename("total")

    state_map = contiguous_us.merge(
        totals, left_on="NAME", right_index=True, how="left"
    )
    state_map["total"] = state_map["total"].fillna(0)

    state_map.plot(
        column="total",
        scheme="quantiles",
        k=5,
        cmap="RdPu",
        edgecolor="gray",
        linewidth=0.5,
        legend=True,
        legend_kwds={"title": "Observation records", "fontsize": 8},
        ax=ax
    )

    ax.set_title("Quantiles")
    ax.set_axis_off()
    return ax

def choroplethNatural(ax, contiguous_us, counts):
    totals = counts.sum(axis=1).rename("total")

    state_map = contiguous_us.merge(
        totals, left_on="NAME", right_index=True, how="left"
    )
    state_map["total"] = state_map["total"].fillna(0)

    state_map.plot(
        column="total",
        scheme="fisherjenks",
        k=5,
        cmap="RdPu",
        edgecolor="gray",
        linewidth=0.5,
        legend=True,
        legend_kwds={"title": "Observation records", "fontsize": 8},
        ax=ax
    )

    ax.set_title("Natural breaks (Fisher–Jenks)")
    ax.set_axis_off()
    return ax

In [ ]:
fig, axes = plt.subplots(
    2, 2,
    figsize=(18, 12),
    layout="constrained"
)

choroplethMap(axes[0, 0], contiguous_us, counts)
axes[0, 0].set_title("Continuous color scale")

choroplethEqual(axes[0, 1], contiguous_us, counts)
choroplethQuantiles(axes[1, 0], contiguous_us, counts)
choroplethNatural(axes[1, 1], contiguous_us, counts)

# fig.savefig("mulberry_classification.png", dpi=300, bbox_inches="tight")
# fig.savefig("mulberry_classification.pdf", bbox_inches="tight")

plt.show()

In [ ]:
## Bivariate plot code generated using AI

import numpy as np
from matplotlib.patches import Rectangle

# Attach each species' counts to the state polygons.
bivariate = contiguous_us.merge(
    counts,
    left_on="NAME",
    right_index=True,
    how="left"
)

species = ["Morus alba", "Morus rubra"]
bivariate[species] = bivariate[species].fillna(0)

# Classify each species:
# 0 = no records
# 1 = positive count at or below the positive-count median
# 2 = count above that median
for name in species:
    values = bivariate[name]
    median = values.loc[values > 0].median()

    bivariate[name + "_class"] = np.select(
        [values == 0, values <= median],
        [0, 1],
        default=2
    )

    print(name, "positive-count median:", median)

# Rows represent increasing red mulberry counts.
# Columns represent increasing white mulberry counts.
palette = np.array([
    ["#eeeeee", "#c5b5d8", "#80619e"],
    ["#e6b899", "#b992a8", "#785b82"],
    ["#cf763e", "#a35e63", "#63384f"]
])

bivariate["color"] = [
    palette[red, white]
    for red, white in zip(
        bivariate["Morus rubra_class"],
        bivariate["Morus alba_class"]
    )
]

fig, ax = plt.subplots(figsize=(13, 8))

bivariate.plot(
    color=bivariate["color"],
    edgecolor="white",
    linewidth=0.5,
    ax=ax
)

ax.set_title("White and red mulberry observations by state")
ax.set_axis_off()

# Add a 3 × 3 bivariate legend.
legend_ax = ax.inset_axes([0.05, 0.05, 0.19, 0.23])

for red in range(3):
    for white in range(3):
        legend_ax.add_patch(
            Rectangle(
                (white, red), 1, 1,
                facecolor=palette[red, white],
                edgecolor="white"
            )
        )

legend_ax.set_xlim(0, 3)
legend_ax.set_ylim(0, 3)
legend_ax.set_aspect("equal")

legend_ax.set_xticks([0.5, 1.5, 2.5])
legend_ax.set_yticks([0.5, 1.5, 2.5])
legend_ax.set_xticklabels(["Zero", "Lower", "Higher"], fontsize=8)
legend_ax.set_yticklabels(["Zero", "Lower", "Higher"], fontsize=8)

legend_ax.set_xlabel("White mulberry", fontsize=9)
legend_ax.set_ylabel("Red mulberry", fontsize=9)
legend_ax.tick_params(length=0)

plt.show()

# Let's make a proportional symbol map 

In [ ]:
## First let's make sure that our statemap has the total mulberry counts 
totals = counts.sum(axis=1).rename("total")

state_map = contiguous_us.merge(
    totals, left_on="NAME", right_index=True, how="left"
)
state_map["total"] = state_map["total"].fillna(0)
state_map.head()

In [ ]:
# Then let's convert those polygons into points
centers = state_map.geometry.centroid
centers.head()

In [ ]:
centers.plot()

In [ ]:

# Let's make a map of the centroids and the states just to make sure that everything is looking ok
# Code partially done with Ai 

def pointMap(ax):

    # The state boundary map
    state_map.plot(
        ax=ax,
        facecolor="whitesmoke",
        edgecolor="gray",
        linewidth=0.5
    )


    # The point map
    ax.scatter(
        centers.x,
        centers.y,
        s=state_map["total"], ## This is the magical line that is making a point map be a proportional size symbol
        color="#8e235c",
        alpha=0.6,
        edgecolors="white",
        linewidths=0.5
    )
    

    ax.legend(
        title="Observation records",
        loc="lower left",
        labelspacing=1.5
    )

    ax.set_title("Mulberry observations by state")
    ax.set_axis_off()

    return ax

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6)) 
ax =  pointMap(ax)
plt.show()

## A proportional map is usually easier to read if it has a few categories.

In [ ]:
#Code using AI

def propSymbol_class(ax):
    
    positive = state_map.loc[state_map["total"] > 0].copy()
    centers = positive.geometry.centroid

    classification = mapclassify.FisherJenks(
        positive["total"],
        k=3
    )

    # The state boundary map
    state_map.plot(
        ax=ax,
        facecolor="whitesmoke",
        edgecolor="gray",
        linewidth=0.5
    )

    # Circle areas for low, medium, and high classes.
    sizes = np.array([60, 250, 700])

    ax.scatter(
        centers.x,
        centers.y,
        s=sizes[classification.yb],
        color="#8e235c",
        alpha=0.6,
        edgecolors="white",
        linewidths=0.5
    )

    # Build the legend using the actual count range in each class.
    for class_id, size in enumerate(sizes):
        values = positive.loc[
            classification.yb == class_id, "total"
        ]

        ax.scatter(
            [], [],
            s=size,
            color="#8e235c",
            alpha=0.6,
            edgecolors="white",
            label=f"{values.min():,.0f}–{values.max():,.0f}"
        )

    ax.legend(
        title="Observation records",
        loc="lower left",
        labelspacing=1.5
    )
    return ax

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6)) 
ax =  propSymbol_class(ax)
plt.show()

In [ ]:
# Let's make a proportional symbol using a PNG

In [ ]:
### Code using AI 


from PIL import Image
from matplotlib.offsetbox import OffsetImage, AnnotationBbox

def iconMap(ax, contiguous_us, counts, icon_path):
    totals = counts.sum(axis=1).rename("total")

    state_map = contiguous_us.to_crs(epsg=5070).merge(
        totals,
        left_on="NAME",
        right_index=True,
        how="left"
    )
    state_map["total"] = state_map["total"].fillna(0)

    state_map.plot(
        ax=ax,
        facecolor="whitesmoke",
        edgecolor="gray",
        linewidth=0.5
    )

    # Read the PNG and trim transparent margins.
    image = Image.open(icon_path).convert("RGBA")
    bounds = image.getbbox()
    if bounds is None:
        raise ValueError("The image is entirely transparent.")

    image = image.crop(bounds)
    image.thumbnail((100, 100))
    icon = np.asarray(image)

    centers = state_map.geometry.centroid

    positive = state_map.loc[state_map["total"] > 0].copy()
    centers = positive.geometry.centroid

    classes = mapclassify.FisherJenks(positive["total"], k=3)
    zooms = [0.15, 0.30, 0.50]

    for x, y, class_id in zip(centers.x, centers.y, classes.yb):
        ax.add_artist(
            AnnotationBbox(
                OffsetImage(icon, zoom=zooms[class_id]),
                (x, y),
                frameon=False,
                pad=0,
                zorder=3
            )
        )

    ax.set_title("Mulberry observations by state")
    ax.set_axis_off()
    return ax

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

iconMap(ax, contiguous_us, counts, "mulberry.png")

plt.show()

# Let's put two variables into one symbol 

In [ ]:
### Code generated using AI 

from matplotlib.patches import Patch
from matplotlib.lines import Line2D

def pieMap(ax, contiguous_us, counts):
    species = ["Morus rubra", "Morus alba"]
    colors = ["firebrick", "mediumpurple"]

    state_map = contiguous_us.to_crs(epsg=5070).merge(
        counts[species],
        left_on="NAME",
        right_index=True,
        how="left"
    )

    state_map[species] = state_map[species].fillna(0)
    state_map["total"] = state_map[species].sum(axis=1)

    positive = state_map.loc[state_map["total"] > 0].copy()

    # Three k-means classes, ordered from low to high.
    np.random.seed(42)
    classification = mapclassify.NaturalBreaks(
        positive["total"], k=3
    )
    positive["size_class"] = classification.yb

    state_map.plot(
        ax=ax,
        facecolor="whitesmoke",
        edgecolor="gray",
        linewidth=0.5
    )

    # Preserve the map extent because ax.pie changes it.
    xlim, ylim = ax.get_xlim(), ax.get_ylim()

    radii = [40000, 80000, 120000]  # Map meters

    for _, state in positive.iterrows():
        center = state.geometry.centroid

        ax.pie(
            [state["Morus rubra"], state["Morus alba"]],
            colors=colors,
            center=(center.x, center.y),
            radius=radii[int(state["size_class"])],
            startangle=90,
            wedgeprops={"edgecolor": "white", "linewidth": 0.5}
        )

    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_aspect("equal")

    # Species legend.
    species_legend = ax.legend(
        handles=[
            Patch(color=colors[0], label="Red mulberry"),
            Patch(color=colors[1], label="White mulberry")
        ],
        loc="lower left"
    )
    ax.add_artist(species_legend)

    # Convert map radii to legend marker diameters in points.
    ax.figure.canvas.draw()
    points_per_meter = (
        ax.get_window_extent().width
        / (xlim[1] - xlim[0])
        * 72 / ax.figure.dpi
    )

    handles = []
    for class_id, radius in enumerate(radii):
        values = positive.loc[
            positive["size_class"] == class_id, "total"
        ]

        handles.append(
            Line2D(
                [], [], linestyle="none", marker="o",
                markersize=2 * radius * points_per_meter,
                markerfacecolor="lightgray",
                markeredgecolor="gray",
                label=f"{values.min():,.0f}–{values.max():,.0f}"
            )
        )

    ax.legend(
        handles=handles,
        title="Records — three classes",
        loc="lower right",
        labelspacing=1.5
    )

    ax.set_title("Mulberry observations by state")
    ax.set_axis_off()
    return ax

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

pieMap(ax, contiguous_us, counts)

plt.show()

Let's make a hexagon choropleth map

In [ ]:
### Code generated using AI 


from shapely.geometry import Polygon

# Equal-area projection with coordinates in meters.
states = contiguous_us
points = points_by_state.copy()

# Hexagon size: center to vertex, in meters.
radius = 100000

xmin, ymin, xmax, ymax = states.total_bounds
hexagons = []

for column, x in enumerate(
    np.arange(xmin - radius, xmax + 2 * radius, 1.5 * radius)
):
    for y in np.arange(
        ymin - radius, ymax + 2 * radius, np.sqrt(3) * radius
    ):
        # Offset alternating columns.
        center_y = y + (column % 2) * np.sqrt(3) * radius / 2

        angles = np.arange(6) * np.pi / 3

        hexagons.append(
            Polygon([
                (
                    x + radius * np.cos(angle),
                    center_y + radius * np.sin(angle)
                )
                for angle in angles
            ])
        )

grid = gpd.GeoDataFrame(
    {"hex_id": range(len(hexagons))},
    geometry=hexagons,
    crs=states.crs
)

# Keep hexagons that intersect the contiguous US.
region = states.geometry.union_all()
grid = grid.loc[grid.intersects(region)].copy()

# Assign each observation to a hexagon.
points = points.reset_index(drop=True)
points["point_id"] = points.index

joined = gpd.sjoin(
    points,
    grid[["hex_id", "geometry"]],
    how="inner",
    predicate="intersects"
)

# Count an observation only once if it falls exactly on a shared edge.
joined = (
    joined.sort_values("hex_id")
    .drop_duplicates("point_id")
)

# Count records by hexagon and species.
hex_counts = (
    joined.groupby(["hex_id", "queried_species"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["Morus rubra", "Morus alba"], fill_value=0)
)

grid = grid.merge(
    hex_counts,
    left_on="hex_id",
    right_index=True,
    how="left"
)

species = ["Morus rubra", "Morus alba"]
grid[species] = grid[species].fillna(0)
grid["total"] = grid[species].sum(axis=1)

In [ ]:
# Let's look at that heaxgon grid
grid.plot() 

In [ ]:
# Code generated using AI 

def hexCountMap(ax, grid, states):
    grid.plot(
        ax=ax,
        column="total",
        cmap="RdPu",
        edgecolor="white",
        linewidth=0.2,
        legend=True,
        legend_kwds={"label": "Number of observation records"}
    )

    states.boundary.plot(
        ax=ax,
        color="gray",
        linewidth=0.4
    )

    ax.set_title("Mulberry observations per hexagon")
    ax.set_axis_off()
    return ax

In [ ]:
# Code generated using AI 

from matplotlib.patches import Rectangle

def hexBivariateMap(ax, grid, states):
    mapped = grid.copy()

    for name in ["Morus alba", "Morus rubra"]:
        values = mapped[name]
        median = values.loc[values > 0].median()

        mapped[name + "_class"] = np.select(
            [values == 0, values <= median],
            [0, 1],
            default=2
        )

    # Rows: increasing red mulberry.
    # Columns: increasing white mulberry.
    palette = np.array([
        ["#eeeeee", "#c5b5d8", "#80619e"],
        ["#e6b899", "#b992a8", "#785b82"],
        ["#cf763e", "#a35e63", "#63384f"]
    ])

    mapped["color"] = [
        palette[red, white]
        for red, white in zip(
            mapped["Morus rubra_class"],
            mapped["Morus alba_class"]
        )
    ]

    mapped.plot(
        ax=ax,
        color=mapped["color"],
        edgecolor="white",
        linewidth=0.2
    )

    states.boundary.plot(ax=ax, color="gray", linewidth=0.4)

    legend_ax = ax.inset_axes([0.05, 0.05, 0.18, 0.23])

    for red in range(3):
        for white in range(3):
            legend_ax.add_patch(
                Rectangle(
                    (white, red), 1, 1,
                    facecolor=palette[red, white],
                    edgecolor="white"
                )
            )

    legend_ax.set_xlim(0, 3)
    legend_ax.set_ylim(0, 3)
    legend_ax.set_aspect("equal")
    legend_ax.set_xticks([0.5, 1.5, 2.5])
    legend_ax.set_yticks([0.5, 1.5, 2.5])
    legend_ax.set_xticklabels(["Zero", "Lower", "Higher"], fontsize=7)
    legend_ax.set_yticklabels(["Zero", "Lower", "Higher"], fontsize=7)
    legend_ax.set_xlabel("White mulberry", fontsize=8)
    legend_ax.set_ylabel("Red mulberry", fontsize=8)
    legend_ax.tick_params(length=0)

    ax.set_title("Red and white mulberry observations")
    ax.set_axis_off()
    return ax

In [ ]:
fig, axes = plt.subplots(
    1, 2,
    figsize=(18, 8),
    layout="constrained"
)

hexCountMap(axes[0], grid, states)
hexBivariateMap(axes[1], grid, states)

plt.show()

# Let's make a logarithmic scale since there are a lot of small values.  

In [ ]:
# Code generated using AI 

from matplotlib.colors import LogNorm

def hexCountMapLog(ax, grid, states):
    grid.plot(
        ax=ax,
        facecolor="lightgray",
        edgecolor="white",
        linewidth=0.2
    )

    positive = grid.loc[grid["total"] > 0]

    if not positive.empty:
        positive.plot(
            ax=ax,
            column="total",
            cmap="RdPu",
            norm=LogNorm(
                vmin=1,
                vmax=max(2, positive["total"].max())
            ),
            edgecolor="white",
            linewidth=0.2,
            legend=True,
            legend_kwds={"label": "Observation records (log scale)"}
        )

    # states.boundary.plot(ax=ax, color="gray", linewidth=0.4)

    ax.set_title("Mulberry observations per hexagon")
    ax.set_axis_off()
    return ax

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

hexCountMapLog(ax, grid, states)

plt.show()